For code style, see PEP 8 (stylishly presented in https://pep8.org/)

If I had more time, I would format all my docstrings in NumPy style, as that's what Colour does, and I use Colour a lot.

In [18]:
import numpy as np
import plotly.graph_objects as go

from colour import SpectralDistribution
from colour.colorimetry import SDS_ILLUMINANTS, SDS_LIGHT_SOURCES, SpectralShape


In [19]:
def planck_relative_spd(wavelengths_nm: np.ndarray, temperature_k: float) -> np.ndarray:
    c2 = 1.438776877e-2  # m*K
    w_m = wavelengths_nm * 1e-9
    x = c2 / (w_m * temperature_k)
    spd = (w_m ** -5) / (np.exp(x) - 1.0)
    return spd / np.max(spd)


def gaussian(wavelengths_nm: np.ndarray, center_nm: float, sigma_nm: float, amplitude: float) -> np.ndarray:
    z = (wavelengths_nm - center_nm) / sigma_nm
    return amplitude * np.exp(-0.5 * z * z)


def modeled_fire_spd(
    wavelengths_nm: np.ndarray,
    continuum_mix: float = 0.5,
    line_strength: float = 0.12,
    overall_gain: float = 1.0,
) -> SpectralDistribution:
    low_1500 = planck_relative_spd(wavelengths_nm, 1500.0)
    high_1900 = planck_relative_spd(wavelengths_nm, 1900.0)

    line_basis = (
        gaussian(wavelengths_nm, 589.3, 2.5, 1.0) +
        gaussian(wavelengths_nm, 766.5, 2.0, 0.7) +
        gaussian(wavelengths_nm, 769.9, 2.0, 0.6)
    )
    if np.max(line_basis) > 0:
        line_basis = line_basis / np.max(line_basis)

    mix = np.clip(continuum_mix, 0.0, 1.0)
    continuum = (1.0 - mix) * low_1500 + mix * high_1900
    values = overall_gain * np.maximum(continuum + line_strength * line_basis, 0.0)

    return SpectralDistribution(dict(zip(wavelengths_nm, values)), name="Modeled Fire (default)")


In [20]:
shape = SpectralShape(380, 780, 5)
wavelengths = np.arange(shape.start, shape.end + shape.interval, shape.interval)

sd_d65 = SDS_ILLUMINANTS["D65"].copy().align(shape)
sd_cool_white_fl = SDS_LIGHT_SOURCES["Cool White FL"].copy().align(shape)
sd_mercury = SDS_LIGHT_SOURCES["Mercury"].copy().align(shape)
sd_kinoton = SDS_LIGHT_SOURCES["Kinoton 75P"].copy().align(shape)
sd_fire = modeled_fire_spd(wavelengths, continuum_mix=0.5, line_strength=0.12, overall_gain=1.0).align(shape)


In [21]:
series = [
    ("D65", sd_d65.values),
    ("Modeled Fire (1500K-1900K mix=0.5, lines=0.12)", sd_fire.values),
    ("Cool White FL", sd_cool_white_fl.values),
    ("Mercury", sd_mercury.values),
    ("Kinoton 75P", sd_kinoton.values)
]
    
fig = go.Figure()
for name, values in series:
    max_value = np.max(values)
    normalized = values / max_value if max_value > 0 else values
    fig.add_trace(
        go.Scatter(
            x=wavelengths,
            y=normalized,
            mode="lines",
            name=name,
            hovertemplate="%{x:.0f} nm<br>%{y:.4f}<extra>" + name + "</extra>",
        )
    )

fig.update_layout(
    title="Comparison of Spectral Power Distributions (normalized)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Relative Power (normalized to max=1)",
    template="plotly_white",
    height=620,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()


## Grape Reflectance Example (Mendeley dataset `gjwx64sgkp`)

This section is set up for the *Spectral dataset of grape berries from hyperspectral imaging for maturity monitoring* dataset.

Expected workflow:
1. Download the dataset file from Mendeley and place it in `resources/data/grape/`.
2. Set `GRAPE_DATA_PATH` in the next cell to that file.
3. Run the cell to parse spectra, aggregate by variety, and show reflected SPDs under selected illuminants.


In [22]:
from pathlib import Path
import csv
import re
from collections import Counter

from plotly.subplots import make_subplots
from pcm.paths import DATA_ROOT

GRAPE_DATA_PATH = DATA_ROOT / 'grape_spectra.csv'


def _header_to_wavelength_nm(header: str) -> float | None:
    s = header.strip().lower().replace('nm', '').replace('_', ' ').strip()
    if s.startswith('x.'):
        s = s[2:]
    s = s.replace(',', '.')
    match = re.search(r'[-+]?\d*\.?\d+', s)
    if not match:
        return None
    value = float(match.group(0))
    if 300.0 <= value <= 2500.0:
        return value
    return None


def _parse_float(value: str) -> float:
    text = str(value).strip()
    if text == '':
        return np.nan
    return float(text.replace(',', '.'))


def _fill_nan_linear(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(values)
    if not np.any(finite):
        return values
    if np.all(finite):
        return values

    idx = np.arange(values.size)
    filled = values.copy()
    filled[~finite] = np.interp(idx[~finite], idx[finite], values[finite])
    return filled


def load_grape_reflectance_sds(csv_path: Path) -> list[SpectralDistribution]:
    if not csv_path.exists():
        raise FileNotFoundError(f'Could not find dataset file: {csv_path}')

    sds: list[SpectralDistribution] = []
    name_counts: Counter[str] = Counter()

    with csv_path.open('r', newline='', encoding='utf-8-sig') as f:
        reader = csv.reader(f, delimiter=';')
        header = next(reader)
        header_norm = [h.strip().lower() for h in header]

        row_number_idx = 0 if header and header[0].strip() == '' else None
        variety_idx = header_norm.index('variety') if 'variety' in header_norm else (1 if len(header) > 1 else 0)
        sugar_idx = next((i for i, h in enumerate(header_norm) if 'sugar' in h), None)

        wavelength_cols: list[tuple[int, float]] = []
        for i, h in enumerate(header):
            if i == row_number_idx or i == variety_idx or i == sugar_idx:
                continue
            wl = _header_to_wavelength_nm(h)
            if wl is not None:
                wavelength_cols.append((i, wl))

        if not wavelength_cols:
            raise ValueError('No wavelength columns found in header.')

        wavelengths = np.array([wl for _idx, wl in wavelength_cols], dtype=np.float64)

        for row_idx, row in enumerate(reader, start=2):
            if len(row) <= max(i for i, _ in wavelength_cols):
                continue

            variety = row[variety_idx].strip() if variety_idx < len(row) else ''
            if variety == '':
                variety = f'row_{row_idx}'

            values = np.array([_parse_float(row[i]) for i, _wl in wavelength_cols], dtype=np.float64)
            values = _fill_nan_linear(values)
            if not np.any(np.isfinite(values)):
                continue

            # Keep variety in the display name; add a counter to avoid duplicate legend names.
            name_counts[variety] += 1
            sd_name = variety if name_counts[variety] == 1 else f'{variety} #{name_counts[variety]}'

            sd = SpectralDistribution(dict(zip(wavelengths, values)), name=sd_name)
            sds.append(sd)

    return sds


def wavelength_to_rgb(wavelength_nm: float, gamma: float = 0.8) -> tuple[float, float, float]:
    if wavelength_nm < 380:
        r, g, b = 0.0, 0.0, 0.0
    elif wavelength_nm < 440:
        r, g, b = -(wavelength_nm - 440) / (440 - 380), 0.0, 1.0
    elif wavelength_nm < 490:
        r, g, b = 0.0, (wavelength_nm - 440) / (490 - 440), 1.0
    elif wavelength_nm < 510:
        r, g, b = 0.0, 1.0, -(wavelength_nm - 510) / (510 - 490)
    elif wavelength_nm < 580:
        r, g, b = (wavelength_nm - 510) / (580 - 510), 1.0, 0.0
    elif wavelength_nm < 645:
        r, g, b = 1.0, -(wavelength_nm - 645) / (645 - 580), 0.0
    elif wavelength_nm <= 780:
        r, g, b = 1.0, 0.0, 0.0
    else:
        r, g, b = 0.0, 0.0, 0.0

    if wavelength_nm < 380:
        factor = 0.0
    elif wavelength_nm < 420:
        factor = 0.3 + 0.7 * (wavelength_nm - 380) / (420 - 380)
    elif wavelength_nm < 700:
        factor = 1.0
    elif wavelength_nm <= 780:
        factor = 0.3 + 0.7 * (780 - wavelength_nm) / (780 - 700)
    else:
        factor = 0.0

    def adjust(c):
        return 0.0 if c == 0.0 else (c * factor) ** gamma

    return adjust(r), adjust(g), adjust(b)


def create_spectral_gradient_colorscale(wavelengths_nm: np.ndarray):
    wl = np.asarray(wavelengths_nm, dtype=np.float64)
    wl_min = float(np.min(wl))
    wl_max = float(np.max(wl))
    span = max(wl_max - wl_min, 1e-12)

    colorscale = []
    for w in wl:
        t = (float(w) - wl_min) / span
        r, g, b = wavelength_to_rgb(float(w))
        colorscale.append([t, f'rgb({int(255*r)}, {int(255*g)}, {int(255*b)})'])

    if colorscale[0][0] != 0.0:
        colorscale.insert(0, [0.0, colorscale[0][1]])
    if colorscale[-1][0] != 1.0:
        colorscale.append([1.0, colorscale[-1][1]])

    return colorscale


def first_sample_per_variety(sds: list[SpectralDistribution]) -> list[SpectralDistribution]:
    out = []
    seen = set()
    for sd in sds:
        variety_base = sd.name.split(' #')[0]
        if variety_base in seen:
            continue
        seen.add(variety_base)
        out.append(sd)
    return out


grape_reflectance_sds = load_grape_reflectance_sds(GRAPE_DATA_PATH)
grape_display_sds = first_sample_per_variety(grape_reflectance_sds)

print(f'Loaded {len(grape_reflectance_sds)} grape reflectance SpectralDistribution rows from {GRAPE_DATA_PATH}')
print(f'Using {len(grape_display_sds)} first-per-variety curves for display:')
print([sd.name for sd in grape_display_sds])

illuminants = {
    'D65': sd_d65.values,
    'Modeled Fire': sd_fire.values,
    'Cool White FL': sd_cool_white_fl.values,
    'Mercury': sd_mercury.values,
    'Kinoton 75P': sd_kinoton.values,
}

target_wavelengths = wavelengths.astype(np.float64)

for illum_name, illum_values in illuminants.items():
    colorscale = create_spectral_gradient_colorscale(target_wavelengths)
    fig = make_subplots(rows=2, cols=1, row_heights=[0.90, 0.10], vertical_spacing=0.0, shared_xaxes=True)

    illum = np.asarray(illum_values, dtype=np.float64)
    illum_n = illum / np.max(illum) if np.max(illum) > 0 else illum

    fig.add_trace(
        go.Scatter(x=target_wavelengths, y=illum_n, mode='lines', name=f'{illum_name} (normalized)', line={'color': 'black', 'width': 2}),
        row=1, col=1,
    )

    for sd in grape_display_sds:
        refl_interp = np.interp(target_wavelengths, sd.wavelengths, sd.values, left=np.nan, right=np.nan)
        refl_interp = np.nan_to_num(refl_interp, nan=0.0, posinf=0.0, neginf=0.0)
        reflected = illum_n * np.clip(refl_interp, 0.0, 1.5)

        fig.add_trace(
            go.Scatter(
                x=target_wavelengths,
                y=reflected,
                mode='lines',
                name=f'{sd.name}: {illum_name} × reflectance',
                line={'width': 2},
            ),
            row=1,
            col=1,
        )

    gradient_data = np.linspace(0, 1, len(target_wavelengths)).reshape(1, -1)
    fig.add_trace(
        go.Heatmap(
            x=target_wavelengths,
            y=[0],
            z=gradient_data,
            colorscale=colorscale,
            showscale=False,
            hoverinfo='skip',
        ),
        row=2,
        col=1,
    )

    fig.update_layout(
        title=f'Reflected SPDs from Grape Reflectance under {illum_name}',
        template='plotly_white',
        height=680,
        legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'left', 'x': 0},
    )
    fig.update_yaxes(title_text='Relative Power', row=1, col=1)
    fig.update_yaxes(showticklabels=False, ticks='', row=2, col=1)
    fig.update_xaxes(title_text='Wavelength (nm)', row=2, col=1)

    fig.show()



Loaded 274 grape reflectance SpectralDistribution rows from /usr/local/repos/git/jgoldstone/pcm-dev/resources/data/grape_spectra.csv
Using 3 first-per-variety curves for display:
['SYRAH', 'MAUZAC', 'FER']
